
# 🌤️ Step-by-Step Weather App Guide (Laravel + Next.js)

This guide walks you through building a decoupled weather application with:

- **Frontend:** Next.js + TypeScript + Tailwind CSS + RippleUI  
- **Backend:** Laravel API  
- **Weather Data:** OpenWeatherMap API  

---

## ✅ Step 1: Get OpenWeatherMap API Key

1. Go to [OpenWeatherMap](https://openweathermap.org/api)  
2. Create a free account  
3. Generate an API key  
4. You'll use this in your Laravel `.env` file  

---

## ✅ Step 2: Laravel Backend

### 2.1. Create Laravel Project

```bash
laravel new weather-api
cd weather-api
```

### 2.2. Add API Key to `.env`

```env
OPENWEATHER_API_KEY=8ef31ff1ffd85561a8e8dfc1eb496534
```

### 2.3. Create Weather Controller

```bash
php artisan make:controller WeatherController
```

### 2.4. Add Controller Logic

`app/Http/Controllers/WeatherController.php`

```php
namespace App\Http\Controllers;

use Illuminate\Http\Request;
use Illuminate\Support\Facades\Http;

class WeatherController extends Controller
{
    public function getWeather(Request $request)
    {
        $city = $request->query('city', 'Nairobi');
        $apiKey = env('OPENWEATHER_API_KEY');

        $response = Http::get("https://api.openweathermap.org/data/2.5/weather", [
            'q' => $city,
            'appid' => $apiKey,
            'units' => 'metric'
        ]);

        return response()->json($response->json());
    }
}
```

### 2.5. Add API Route

`routes/api.php`

```php
use App\Http\Controllers\WeatherController;

Route::get('/weather', [WeatherController::class, 'getWeather']);
```

### 2.6. Run the Server

```bash
php artisan serve
```

Test with:  
`http://127.0.0.1:8000/api/weather?city=Nairobi`

---

## ✅ Step 3: Next.js Frontend Setup

### 3.1. Create Project

```bash
npx create-next-app@latest weather-ui --typescript
cd weather-ui
```

### 3.2. Install Tailwind CSS and RippleUI

```bash
npm install -D tailwindcss postcss autoprefixer
npx tailwindcss init -p
npm install rippleui
```

### 3.3. Configure Tailwind

In `tailwind.config.js`:

```js
module.exports = {
  content: [
    "./app/**/*.{js,ts,jsx,tsx}",
    "./pages/**/*.{js,ts,jsx,tsx}",
    "./components/**/*.{js,ts,jsx,tsx}",
    "./node_modules/rippleui/**/*.{js,ts,jsx,tsx}",
  ],
  theme: {
    extend: {},
  },
  plugins: [require("rippleui")],
};
```

In `globals.css`:

```css
@tailwind base;
@tailwind components;
@tailwind utilities;
```

---

## ✅ Step 4: Create UI Components

### 4.1. WeatherCard Component

Create: `components/WeatherCard.tsx`

```tsx
"use client";
import { useEffect, useState } from "react";

interface WeatherData {
  name: string;
  main: { temp: number };
  weather: { description: string; icon: string }[];
}

export default function WeatherCard({ city }: { city: string }) {
  const [data, setData] = useState<WeatherData | null>(null);
  const [loading, setLoading] = useState(true);

  useEffect(() => {
    fetch(`http://127.0.0.1:8000/api/weather?city=${city}`)
      .then((res) => res.json())
      .then((data) => {
        setData(data);
        setLoading(false);
      });
  }, [city]);

  if (loading) return <p className="text-center">Loading...</p>;
  if (!data) return <p className="text-center">No data available</p>;

  return (
    <div className="card w-full bg-base-100 shadow-xl p-4">
      <h2 className="text-xl font-bold">{data.name}</h2>
      <p className="text-lg">{data.main.temp}°C</p>
      <p className="capitalize">{data.weather[0].description}</p>
      <img
        src={`http://openweathermap.org/img/wn/${data.weather[0].icon}@2x.png`}
        alt="weather icon"
      />
    </div>
  );
}
```

---

### 4.2. Update Homepage

Update `app/page.tsx`:

```tsx
"use client";
import { useState } from "react";
import WeatherCard from "../components/WeatherCard";

export default function Home() {
  const [city, setCity] = useState("Nairobi");

  return (
    <main className="p-6 flex flex-col items-center space-y-6">
      <input
        className="input input-bordered w-full max-w-xs"
        type="text"
        placeholder="Enter city"
        value={city}
        onChange={(e) => setCity(e.target.value)}
      />
      <WeatherCard city={city} />
    </main>
  );
}
```

---

## ✅ Step 5: Run and Test

### Backend

```bash
cd weather-api
php artisan serve
```

### Frontend

```bash
cd weather-ui
npm run dev
```

Visit: [http://localhost:3000](http://localhost:3000)

---

## ✅ Bonus Tips

| Feature                  | Recommendation                            |
|--------------------------|-------------------------------------------|
| Type safety              | Use TypeScript interfaces                 |
| API error handling       | Add `try/catch` in `fetch()` logic        |
| UX improvements          | Add loading spinners, fallback messages   |
| Visual polish            | Use RippleUI Cards and Icons              |
| Git hygiene              | Commit messages like `feat: add API call`|

---


